In [1]:
import pandas as pd
import numpy as np
from statsmodels.regression.mixed_linear_model import MixedLM
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.regression.mixed_linear_model import MixedLM
import statsmodels.api as sm
sns.set_style("white")
from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests
import os

In [2]:
# file in phenotypic projection code

SEED = 38
model_type = 'C-GMVAE'
run_where = 'local' # tell if I'm running locally or on Great Lakes

if run_where == 'local':
    file_path = '/home/nghayes/latent-space-trajectories/latent_variables_epoch20000.csv'
elif run_where == 'great-lakes':
    default_path = f'/nfs/turbo/umms-kaczoro/u19-shared/vae-HC-PFC/14mon_ADBXD_joined/seed_model/{model_type}_SEED_{SEED}'
    file_path = os.path.join(default_path, 'latent_variables_epoch20000.csv')

# default_path = f'/nfs/turbo/umms-kaczoro/u19-shared/vae-HC-PFC/14mon_ADBXD_joined/seed_model/{model_type}_SEED_{SEED}'
# default_path = f'L:/yidingca/snRNA_HC_PFC_VAE/trained_model_QRT_prior/{model_type}_SEED_{SEED}'
# file_path = os.path.join(default_path, 'latent_variables_epoch20000.csv')
# file_path = '/home/nghayes/latent-space-trajectories/latent_variables_epoch20000.csv'
df = pd.read_csv(file_path, index_col=0)

In [3]:
df.loc[:20,:]

,LV1,LV2,LV3,LV4,LV5,LV6,LV7,LV8,LV9,LV10,...,cell_type,SCBID,MouseID,Age,Strain,Genotype,CFM_14_5_snRNA,14-6-residual,14-6-Bin,region
0,-24.256855,-18.943563,-19.148005,-13.069660,-7.281853,-6.647669,-1.973320,1.312754,5.479063,10.920240,...,Glut,CK19047,1583,14,33,5,1.047481,0.133901,2,PFC
1,-32.498856,-27.973068,-25.985146,-23.571732,-19.487030,-15.888780,-14.033054,-9.226448,-8.665678,-4.734122,...,Gaba,CK19032,1596,14,124,5,1.231932,0.738297,3,PFC
2,-17.562922,-16.157148,-7.969545,-5.897250,1.853780,4.455924,9.762284,14.726957,23.734491,29.096786,...,Astrocytes,CK19107,1583,14,33,5,1.047481,0.383536,2,HC
3,-28.858011,-24.174309,-22.691444,-19.941648,-16.066748,-12.556651,-10.494271,-5.782712,-5.136093,-1.035175,...,glut,CK19119,5696,14,B6,5,-0.825919,1.040817,3,HC
4,-10.571421,-6.463661,0.640753,7.082870,11.920109,17.315226,24.198599,28.219280,36.607440,41.267610,...,glut,CK19120,1532,14,65,5,0.708210,-1.314067,0,HC
5,-8.174380,-4.043670,2.960599,9.497574,14.260602,19.690483,26.765537,30.756020,39.044262,43.966454,...,glut,CK19021,2105,14,161,5,-1.625947,-1.269611,0,HC
6,-12.695596,-7.713936,-4.878980,-1.440421,6.012704,8.817274,13.797771,18.695784,27.195430,32.340393,...,Immune,CK19012,1576,14,D2,5,-1.340011,-0.673075,1,HC
7,-16.000040,-10.902713,-6.684124,-3.191145,4.525127,6.925598,12.219912,16.805430,25.249464,30.431843,...,glut,CK19014,2644,14,50,5,-1.073335,-0.777041,1,HC
8,-29.944551,-25.133204,-23.212881,-20.931145,-16.555386,-13.060875,-11.124614,-6.219855,-5.710696,-1.613443,...,glut,CK19078,1596,14,124,5,1.231932,1.277393,3,HC
9,-24.174894,-18.247770,-17.225182,-11.089184,-5.395521,-4.119117,0.220880,4.212738,8.117303,13.438895,...,Gaba,CK19063,2433,14,113,5,0.550426,-0.148236,2,PFC


In [4]:
# check residual values and mouse IDs
print('unique residual values: ', df['14-6-residual'].unique()) # there are 25 unique values but 13 strains??
print('unique mouse IDs: ', df['MouseID'].unique())
print('unique strains: ', df['Strain'].unique())

unique residual values:  [ 0.13390099  0.7382975   0.38353629  1.04081711 -1.31406651 -1.26961126
 -0.67307462 -0.77704088  1.2773932  -0.14823589 -0.55246417  0.58568163
 -1.36427551  0.66093217 -0.25260968 -0.11113249 -0.98984271 -0.57673104
 -0.97414041  1.04391985 -0.92848383 -0.62782032 -0.46046176 -0.23483939
  0.58351163]
unique mouse IDs:  [1583 1596 5696 1532 2105 1576 2644 2433 2648 1966 1272 1482 1490]
unique strains:  <StringArray>
['33', '124', 'B6', '65', '161', 'D2', '50', '113', '99', '83', '53', '39',
 '28']
Length: 13, dtype: str


In [5]:
# check all entries for particular mouse ID (for one strain)
df[df['MouseID']==1583]

,LV1,LV2,LV3,LV4,LV5,LV6,LV7,LV8,LV9,LV10,...,cell_type,SCBID,MouseID,Age,Strain,Genotype,CFM_14_5_snRNA,14-6-residual,14-6-Bin,region
0,-24.256855,-18.943563,-19.148005,-13.069660,-7.281853,-6.647669,-1.973320,1.312754,5.479063,10.920240,...,Glut,CK19047,1583,14,33,5,1.047481,0.133901,2,PFC
2,-17.562922,-16.157148,-7.969545,-5.897250,1.853780,4.455924,9.762284,14.726957,23.734491,29.096786,...,Astrocytes,CK19107,1583,14,33,5,1.047481,0.383536,2,HC
20,-23.562283,-20.117138,-16.579475,-12.704189,-6.931943,-5.821667,-1.278652,2.270180,6.310486,11.782984,...,Gaba,CK19047,1583,14,33,5,1.047481,0.133901,2,PFC
22,-23.363234,-19.062843,-16.351782,-11.500536,-5.752747,-4.625563,-0.061926,3.472979,7.540337,13.102202,...,Gaba,CK19047,1583,14,33,5,1.047481,0.133901,2,PFC
28,-23.182390,-20.512297,-17.084257,-13.049519,-7.131986,-6.267180,-1.922619,1.612668,5.320275,10.894245,...,glut,CK19107,1583,14,33,5,1.047481,0.383536,2,HC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56621,-18.056065,-16.248060,-8.343041,-5.906617,1.762480,4.462933,9.783082,14.553792,23.329315,28.548754,...,Oligodendrocytes,CK19107,1583,14,33,5,1.047481,0.383536,2,HC
56624,-23.023726,-22.127516,-16.987013,-14.011026,-8.314982,-7.162544,-3.398485,0.485144,3.436814,8.758227,...,glut,CK19107,1583,14,33,5,1.047481,0.383536,2,HC
56626,-17.122196,-15.821091,-8.365338,-5.468097,2.618603,4.962369,10.304828,15.051514,23.456959,28.764640,...,glut,CK19107,1583,14,33,5,1.047481,0.383536,2,HC
56627,-23.568130,-22.981295,-17.807919,-14.080692,-8.169436,-6.616759,-2.796304,1.698078,4.619236,9.929455,...,Gaba,CK19047,1583,14,33,5,1.047481,0.133901,2,PFC


In [13]:
# df: starting with latent space data, which will get edited with PP info
# want to create two new data frames:
# df_latent: a new data frame from df, which will end up with decoded cfm info
# df_recons: a new data frame from the reconstruction data
LATENT_DIM = 10
RECONS_DATA_CSV  = '/home/nghayes/latent-space-trajectories/recons_cfm.csv'

print(df['14-6-Bin'].value_counts())
print(df.describe())
df_latent = df.copy()
cols = list(df_latent.columns[:LATENT_DIM]) + ["14-6-Bin"] # only want 10 LVs + which bin each observation is in
df_latent = df_latent[cols].copy()
df_latent.columns = [f"z_{i+1}" for i in range(LATENT_DIM)] + ["label"] # changing names of columns to z_i

df_recons = pd.read_csv(RECONS_DATA_CSV, index_col=0)
print(df_recons['epoch'].unique()) # ALERT: mixed type for epoch column - not problem for me now, but could be later
df_recons = df_recons.loc[df_recons['epoch'] == 20000] # I added this to choose last epoch
df_latent['decoded_cfm'] = df_recons['cfm'].values # creating new column with decoded cfm values from recons csv

14-6-Bin
2    20105
3    13668
1    11443
0    11434
Name: count, dtype: int64
                LV1           LV2           LV3           LV4           LV5  \
count  56650.000000  56650.000000  56650.000000  56650.000000  56650.000000   
mean     -20.256705    -15.159655    -11.679939     -6.995157     -1.338052   
std        8.119472      7.786041      9.916827     10.911846     11.390052   
min      -34.214813    -29.359503    -27.684828    -25.183409    -21.168283   
25%      -27.479808    -21.779699    -20.591530    -15.195161     -9.749770   
50%      -20.592559    -14.820463    -12.114686     -7.309843     -0.326291   
75%      -13.231291     -8.528811     -3.410012      0.348975      7.869470   
max       -5.473288     -1.340881      5.552579     12.048773     16.966198   

                LV6           LV7           LV8           LV9          LV10  \
count  56650.000000  56650.000000  56650.000000  56650.000000  56650.000000   
mean       1.713566      6.359805     10.704072    

/tmp/ipykernel_11612/1256302770.py:15: DtypeWarning: Columns (0: Strain) have mixed types. Specify dtype option on import or set low_memory=False.
  df_recons = pd.read_csv(RECONS_DATA_CSV, index_col=0)


In [17]:
df

,LV1,LV2,LV3,LV4,LV5,LV6,LV7,LV8,LV9,LV10,...,cell_type,SCBID,MouseID,Age,Strain,Genotype,CFM_14_5_snRNA,14-6-residual,14-6-Bin,region
0,-24.256855,-18.943563,-19.148005,-13.069660,-7.281853,-6.647669,-1.973320,1.312754,5.479063,10.920240,...,Glut,CK19047,1583,14,33,5,1.047481,0.133901,2,PFC
1,-32.498856,-27.973068,-25.985146,-23.571732,-19.487030,-15.888780,-14.033054,-9.226448,-8.665678,-4.734122,...,Gaba,CK19032,1596,14,124,5,1.231932,0.738297,3,PFC
2,-17.562922,-16.157148,-7.969545,-5.897250,1.853780,4.455924,9.762284,14.726957,23.734491,29.096786,...,Astrocytes,CK19107,1583,14,33,5,1.047481,0.383536,2,HC
3,-28.858011,-24.174309,-22.691444,-19.941648,-16.066748,-12.556651,-10.494271,-5.782712,-5.136093,-1.035175,...,glut,CK19119,5696,14,B6,5,-0.825919,1.040817,3,HC
4,-10.571421,-6.463661,0.640753,7.082870,11.920109,17.315226,24.198599,28.219280,36.607440,41.267610,...,glut,CK19120,1532,14,65,5,0.708210,-1.314067,0,HC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56645,-28.579952,-23.901604,-22.118683,-19.591473,-15.684873,-12.046993,-10.108648,-5.416104,-4.710186,-0.721344,...,Oligo,CK19111,1272,14,53,5,0.263749,0.660932,3,PFC
56646,-7.582398,-3.366018,3.963249,10.430345,15.107400,20.847500,27.655052,31.937830,40.201540,44.993100,...,Gaba,CK19033,1966,14,83,5,-0.974072,-0.989843,0,PFC
56647,-17.119564,-13.705447,-7.196110,-4.540887,3.223161,5.727212,10.905486,15.751303,24.399230,29.637558,...,Astrocytes,CK19059,2433,14,113,5,0.550426,-0.111132,2,HC
56648,-29.270935,-24.531736,-22.611712,-20.303423,-16.054407,-12.572237,-10.648892,-5.845601,-5.259646,-1.332067,...,glut,CK19078,1596,14,124,5,1.231932,1.277393,3,HC


In [15]:
df_recons

,cfm,epoch,Strain
1133000,1.048495,20000,33
1133001,1.236806,20000,124
1133002,1.044764,20000,33
1133003,-0.825561,20000,B6
1133004,0.690024,20000,65
...,...,...,...
1189645,0.267969,20000,53
1189646,-0.972245,20000,83
1189647,0.547159,20000,113
1189648,1.229240,20000,124


In [16]:
df_latent

,z_1,z_2,z_3,z_4,z_5,z_6,z_7,z_8,z_9,z_10,label,decoded_cfm
0,-24.256855,-18.943563,-19.148005,-13.069660,-7.281853,-6.647669,-1.973320,1.312754,5.479063,10.920240,2,1.048495
1,-32.498856,-27.973068,-25.985146,-23.571732,-19.487030,-15.888780,-14.033054,-9.226448,-8.665678,-4.734122,3,1.236806
2,-17.562922,-16.157148,-7.969545,-5.897250,1.853780,4.455924,9.762284,14.726957,23.734491,29.096786,2,1.044764
3,-28.858011,-24.174309,-22.691444,-19.941648,-16.066748,-12.556651,-10.494271,-5.782712,-5.136093,-1.035175,3,-0.825561
4,-10.571421,-6.463661,0.640753,7.082870,11.920109,17.315226,24.198599,28.219280,36.607440,41.267610,0,0.690024
...,...,...,...,...,...,...,...,...,...,...,...,...
56645,-28.579952,-23.901604,-22.118683,-19.591473,-15.684873,-12.046993,-10.108648,-5.416104,-4.710186,-0.721344,3,0.267969
56646,-7.582398,-3.366018,3.963249,10.430345,15.107400,20.847500,27.655052,31.937830,40.201540,44.993100,0,-0.972245
56647,-17.119564,-13.705447,-7.196110,-4.540887,3.223161,5.727212,10.905486,15.751303,24.399230,29.637558,2,0.547159
56648,-29.270935,-24.531736,-22.611712,-20.303423,-16.054407,-12.572237,-10.648892,-5.845601,-5.259646,-1.332067,3,1.229240
